In [56]:
import numpy as np
import pandas as pd
import sklearn


In [57]:
df_train = pd.read_csv('../data/processed/train_data.csv')
df_test = pd.read_csv('../data/processed/test_data.csv')

In [58]:
df_test.head(1)

,year,month,day,hour,PM2.5,TEMP,PRES,DEWP,RAIN,wd,...,NO2_shift24,CO_shift24,O3_shift24,WSPM_shift24,TEMP_shift24,DEWP_shift24,PM2.5_rolling_mean_24,WSPM_rolling_mean_24,PM2.5_trend,Temp_Dewp_Diff
0,2017,1,1,0,485.0,-4.7,1022.1,-6.1,0.0,ENE,...,144.0,5100.0,3.0,1.0,-5.0,-7.9,200.458333,1.066667,-3.0,2.9


In [59]:
X_train = df_train.drop(columns=['PM2.5'])
y_train = df_train['PM2.5']
X_test = df_test.drop(columns=['PM2.5'])
y_test = df_test['PM2.5']

In [60]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

cat_col = ['station','wd']
num_col = [col for col in X_train.columns if col not in cat_col] 

processing = ColumnTransformer(transformers=[
    ('encode',OneHotEncoder(drop='first',sparse_output=False),cat_col),
    ('scale',StandardScaler(),num_col)
])

In [61]:
X_train_processed = processing.fit_transform(X_train)
X_test_processed = processing.transform(X_test)

In [62]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [63]:
from sklearn.metrics import mean_squared_error, r2_score,root_mean_squared_error
import time

import warnings
warnings.filterwarnings('ignore')

In [64]:
modellar = {
    'Liner Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'ElasticNet Regression': ElasticNet(),
    'SGD Regression': SGDRegressor(),
    'Decision Tree Regression': DecisionTreeRegressor(max_depth=15),
    'Linear SVR': LinearSVR(),
    'KNN Regression': KNeighborsRegressor(),
    'Random Forest Regression': RandomForestRegressor(n_estimators=70,n_jobs=-1,max_depth=15),
    'Extra Trees Regression': ExtraTreesRegressor(n_estimators=70,n_jobs=-1,max_depth=15),
    'Hist Gradient Boosting Regression': HistGradientBoostingRegressor(),
    #'MLP Regression': MLPRegressor(max_iter=500),
    
    'XGBoost Regression': XGBRegressor(),
    'LightGBM Regression': LGBMRegressor(),
    'CatBoost Regression': CatBoostRegressor(verbose=0)
}

In [65]:
natijalar = []

for nomi, model in modellar.items():
    print(f"⏳ {nomi} o'qitilmoqda...")
    start_time = time.time()
    
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    lag_time = time.time() - start_time
    
    natijalar.append({
        'Model': nomi,
        'MSE': round(mse, 3),
        'RMSE': round(rmse, 3),
        'R2 Score': round(r2, 3),
        'Ketgan vaqt (sek)': round(lag_time, 3)
    })
    
df_natija = pd.DataFrame(natijalar)

⏳ Liner Regression o'qitilmoqda...
⏳ Ridge Regression o'qitilmoqda...
⏳ Lasso Regression o'qitilmoqda...
⏳ ElasticNet Regression o'qitilmoqda...
⏳ SGD Regression o'qitilmoqda...
⏳ Decision Tree Regression o'qitilmoqda...
⏳ Linear SVR o'qitilmoqda...
⏳ KNN Regression o'qitilmoqda...
⏳ Random Forest Regression o'qitilmoqda...
⏳ Extra Trees Regression o'qitilmoqda...
⏳ Hist Gradient Boosting Regression o'qitilmoqda...
⏳ XGBoost Regression o'qitilmoqda...
⏳ LightGBM Regression o'qitilmoqda...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005445 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4440
[LightGBM] [Info] Number of data points in the train set: 403200, number of used features: 48
[LightGBM] [Info] Start training from score 79.315063
⏳ CatBoost Regression o'qitilmoqda...


In [66]:
df_natija.sort_values(by='R2 Score', ascending=False)

,Model,MSE,RMSE,R2 Score,Ketgan vaqt (sek)
12,LightGBM Regression,5074.463,71.235,0.596,1.415
8,Random Forest Regression,5087.712,71.328,0.595,68.809
10,Hist Gradient Boosting Regression,5118.779,71.546,0.592,3.940
9,Extra Trees Regression,5157.835,71.818,0.589,69.076
11,XGBoost Regression,5365.032,73.246,0.573,1.650
13,CatBoost Regression,5463.608,73.916,0.565,26.743
4,SGD Regression,7124.863,84.409,0.433,4.225
0,Liner Regression,7132.074,84.452,0.432,0.779
1,Ridge Regression,7132.078,84.452,0.432,0.140
2,Lasso Regression,7365.123,85.820,0.414,1.556
